# Practical Assignment 11: Image Denoising using Autoencoders on MNIST

**Aim:** Implement the concept of image denoising using autoencoders on the MNIST handwritten digit dataset.

**Key Concepts to Remember:**
1. **Noise Generation:** Corrupt clean images with additive Gaussian noise ($X_{\text{noisy}} = X_{\text{clean}} + \sigma \cdot \mathcal{N}(0, 1)$).
2. **Denoising Training Trick:** Feed $X_{\text{noisy}}$ as input, but use $X_{\text{clean}}$ as target (`model.fit(X_noisy, X_clean)`).
3. **Result:** The network learns the underlying manifold and outputs reconstructed noise-free images.


In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

# Step 1: Load Clean Images and Synthesize Corrupted Noisy Images
df = pd.read_csv(r'd:/workspace01/4th year/DL/LabPracticesIV_DEEP_LEARNING/dataset/MNIST/mnist_784_csv.csv', nrows=6000)
X_clean = df.iloc[:, :-1].values.astype('float32') / 255.0
X_train_clean, X_test_clean = train_test_split(X_clean, test_size=0.2, random_state=42)

# Add synthetic Gaussian noise with noise factor 0.35
noise_factor = 0.35
np.random.seed(42)
X_train_noisy = np.clip(X_train_clean + noise_factor * np.random.normal(size=X_train_clean.shape), 0.0, 1.0)
X_test_noisy = np.clip(X_test_clean + noise_factor * np.random.normal(size=X_test_clean.shape), 0.0, 1.0)

print('Original clean shape:', X_train_clean.shape)
print('Initial Corrupted Noisy Test MSE:', np.mean((X_test_noisy - X_test_clean)**2))

# Step 2: Build Denoising Autoencoder Architecture
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(784,)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(64, activation='relu'),  # Bottleneck
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(784, activation='sigmoid')
])
model.summary()

# Step 3: Train Model (Input = Noisy, Target = Clean!)
model.compile(optimizer='adam', loss='binary_crossentropy')
history = model.fit(X_train_noisy, X_train_clean, epochs=5, batch_size=64, validation_data=(X_test_noisy, X_test_clean), verbose=1)

# Step 4: Evaluate Denoising Performance
denoised_imgs = model.predict(X_test_noisy, verbose=0)
initial_err = np.mean((X_test_noisy - X_test_clean)**2)
final_err = np.mean((denoised_imgs - X_test_clean)**2)
reduction = ((initial_err - final_err) / initial_err) * 100

print(f'\nInitial Corrupted Images MSE: {initial_err:.5f}')
print(f'Restored Denoised Images MSE: {final_err:.5f}')
print(f'Noise Reduction Improvement:  {reduction:.2f}% (Autoencoder successfully restored clean digits)')


Original clean shape: (4800, 784)
Initial Corrupted Noisy Test MSE: 0.06259677050758075
Model: "sequential_8"
┌─────────────────────────────────┬────────────────────────┬───────────────┐
│ Layer (type)                    │ Output Shape           │       Param # │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_24 (Dense)                │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_25 (Dense)                │ (None, 784)            │       101,136 │
└─────────────────────────────────┴────────────────────────┴───────────────┘
 Total params: 218,192 (852.31 KB)
 Trainab